# 02. 변화율 모형 — 오전 · 종일

## 변화율 모형이란
- 식: `log(2025년 승차) = log(2024년 승차) + β·X`
  - `log(2024년 승차)` 항(**offset**)은 계수가 1로 고정되어 있다. 그래서 β는 **2025/2024 비율(변화율)이 X에 따라 어떻게 달라지는가**를 뜻한다.
- 셀 규모는 자동으로 통제된다. 100명 → 80명인 셀과 10명 → 8명인 셀을 똑같이 −20%로 본다.
- 승차인원은 0 이상의 정수(카운트)이므로 **Poisson 회귀 + 강건 표준오차(HC1)**를 쓴다.
  - 이 방식(QMLE)은 분산 가정이 틀려도 계수가 일관되게 추정된다.
  - 비교용으로 **음이항(NB)** 회귀도 같이 돌린다.
- **계수 읽는 법**: `exp(β) − 1` = 변화율의 상대 차이. 표의 `변화율 차이(%)` 열이 이 값이다.
- 2024년 승차가 0명인 셀은 비율을 정의할 수 없어서 제외된다 (오전 118개, 종일 132개).

## 왜 종일도 보나
데이터가 연도별 하루치라 오전 승차가 몇 명뿐인 셀이 많다. 같은 날의 00~23시 전체를 쓰면 셀당 표본이 약 4배가 되어 잡음이 줄어든다(강건성 검토).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # 정리본 최상위의 config.py

from config import *
setup_notebook()

import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy import stats

data = load_model_data()  # data/processed/model_dataset.csv

In [2]:
pd.DataFrame({
    "07~09시": [data.board_2024.sum(), data.board_2025.sum(), data.board_2024.median(), (data.board_2024 < 5).sum(), (data.board_2024 == 0).sum()],
    "종일": [data.board_2024_all.sum(), data.board_2025_all.sum(), data.board_2024_all.median(), (data.board_2024_all < 5).sum(), (data.board_2024_all == 0).sum()],
}, index=["2024년 합계", "2025년 합계", "2024년 셀당 중앙값", "2024년 5명 미만 셀", "2024년 0명 셀"]).round(1)

,07~09시,종일
2024년 합계,6044.0,19884.0
2025년 합계,4757.0,16300.0
2024년 셀당 중앙값,17.5,67.0
2024년 5명 미만 셀,31.0,14.0
2024년 0명 셀,14.0,0.0


In [3]:
results = {
    "오전 Poisson": rate_model(data, "board_2025", "board_2024", X_MAIN),
    "오전 NB": rate_model(data, "board_2025", "board_2024", X_MAIN, "nb"),
    "종일 Poisson": rate_model(data, "board_2025_all", "board_2024_all", X_MAIN),
    "종일 NB": rate_model(data, "board_2025_all", "board_2024_all", X_MAIN, "nb"),
}
pd.concat({name: coef_table(r)[["coef", "p", "유의"]] for name, r in results.items()}, axis=1)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


오전 Poisson                오전 NB             종일 Poisson                종일 NB             
                       coef       p   유의    coef       p  유의       coef       p   유의    coef       p   유의
const               -0.0552  0.7469       0.2550  0.2466        -0.2526  0.2331      -0.0448  0.7602     
GTX역 보행시간(분)         0.0050  0.0213   **  0.0010  0.6221         0.0048  0.0021  ***  0.0049  0.0001  ***
IC까지 도로거리(km)       -0.1017  0.0018  *** -0.0674  0.0341  **    -0.0502  0.0699    * -0.0391  0.0676    *
GTX 경쟁 목적지 비율(%)    -0.0013  0.7245       0.0016  0.6288        -0.0040  0.1943      -0.0009  0.6953     
2024년 20~50대 인구      0.0001  0.3445      -0.0001  0.3311         0.0002  0.0499   ** -0.0000  0.9117     
2024년 건축물 수          0.0004  0.8902       0.0010  0.6866        -0.0000  0.9912      -0.0024  0.1495

In [4]:
# 보행시간 10분당 변화율 차이
for name, r in results.items():
    print(f"{name}: 보행시간 10분 늘 때 2025/2024 비율 {(np.exp(r.params['walk_min'] * 10) - 1) * 100:+.1f}%  (p={r.pvalues['walk_min']:.4f})")

오전 Poisson: 보행시간 10분 늘 때 2025/2024 비율 +5.1%  (p=0.0213)
오전 NB: 보행시간 10분 늘 때 2025/2024 비율 +1.1%  (p=0.6221)
종일 Poisson: 보행시간 10분 늘 때 2025/2024 비율 +4.9%  (p=0.0021)
종일 NB: 보행시간 10분 늘 때 2025/2024 비율 +5.0%  (p=0.0001)


## 결론
- **GTX 보행시간(+)**: 역에서 멀수록 감소율이 작다 = **역에 가까울수록 더 많이 줄었다**.
  - 종일 기준 Poisson·NB 모두 p<0.01이고, 10분당 약 +5%다.
  - 오전 기준은 Poisson에서만 유의하다. 오전 단독으로는 아직 불안정하다.
- **IC까지 도로거리(−)**: 네 모형 모두 같은 방향이다(p<0.1). **IC에서 먼 셀일수록 감소율이 컸다.**
- 경쟁 목적지 비율과 건축물 수는 어떤 모형에서도 유의하지 않다.
- 보행시간 효과를 구간으로 나눠 더 직접적으로 보는 분석 → `03_보행시간_구간분석.ipynb`